In [1]:
import psycopg2
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

print("Connecting to data warehouse for Revenue Cycle Analytics...")

# 1. FETCH INTEGRATED RCM DATA
conn = psycopg2.connect(
    host="localhost", database="HealthHub Star Schema Data Warehouse", user="postgres", password="admin123"
)

# Crucial: We must join billing facts with encounter fields to get the ICD-10 diagnostic context
query = """
SELECT 
    b.CPT_Code, b.Insurance_Carrier, b.Gross_Amount_AED, b.Claim_Status,
    e.Clinic_Specialty, e.ICD10_Code
FROM fact_billing b
JOIN fact_encounters e ON b.Encounter_ID = e.Encounter_ID;
"""
df = pd.read_sql(query, conn)
conn.close()

# Convert target column into a binary class: 1 if Rejected (Risk), 0 if Approved
df['is_rejected'] = df['claim_status'].apply(lambda x: 1 if x == 'Rejected' else 0)
df = df.drop(columns=['claim_status'])

# 2. ADVANCED ENCODING FOR CATEGORICAL HEALTHCARE VALUES
# High cardinality categorical variables (like unique ICD/CPT pairs) require clear encoding 
X = df.drop(columns=['is_rejected'])
y = df['is_rejected']

categorical_features = ['cpt_code', 'insurance_carrier', 'clinic_specialty', 'icd10_code']
numeric_features = ['gross_amount_aed']

# Use ColumnTransformer to neatly handle pipeline preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ], remainder='passthrough'
)

X_processed = preprocessor.fit_transform(X)

# 3. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42, stratify=y)

# 4. TRAINING THE RCM ENGINE
# We adjust 'scale_pos_weight' because claim rejections are typically minority imbalances
print("Training Claim Denial Risk Engine...")
rcm_model = XGBClassifier(n_estimators=150, max_depth=4, scale_pos_weight=2, random_state=42)
rcm_model.fit(X_train, y_train)

# 5. BUSINESS METRIC EVALUATION
y_pred = rcm_model.predict(X_test)

print("\n FINANCIAL RCM EVALUATION FRAMEWORK SUMMARY:")
print(classification_report(y_test, y_pred, target_names=['Approved (Safe)', 'Rejected (Risk)']))

print("CONFUSION MATRIX ANALYSIS FOR THE CFO:")
cm = confusion_matrix(y_test, y_pred)
print(f"True Safe Claims (Correctly identified): {cm[0][0]}")
print(f"Missed Financial Leakages (False Negatives - BAD): {cm[1][0]}")
print(f"Caught Rejections Before Submission (True Positives): {cm[1][1]}")


Connecting to data warehouse for Revenue Cycle Analytics...


C:\Users\USER\AppData\Local\Temp\ipykernel_17336\4031445843.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Training Claim Denial Risk Engine...

 FINANCIAL RCM EVALUATION FRAMEWORK SUMMARY:
                 precision    recall  f1-score   support

Approved (Safe)       0.79      0.81      0.80       124
Rejected (Risk)       0.30      0.27      0.29        37

       accuracy                           0.69       161
      macro avg       0.55      0.54      0.54       161
   weighted avg       0.68      0.69      0.68       161

CONFUSION MATRIX ANALYSIS FOR THE CFO:
True Safe Claims (Correctly identified): 101
Missed Financial Leakages (False Negatives - BAD): 27
Caught Rejections Before Submission (True Positives): 10
